# YGP区域Sentinel-2年度合成与分块导出
（1）读取本地研究区和分幅瓦片数据，按编号筛选待导出的分幅瓦片。   
（2）筛选年度 Sentinel-2 影像，关联 Cloud Score+，保留清晰像元及六个目标波段。   
（3）分别计算秋季（9—11 月）和其他月份的中值合成，以其他月份补齐秋季空缺，无数据像元填 0。   
（4）数据可视化    
（5）数据导出  


In [17]:
import os
import ee
import geemap 
from pathlib import Path  


In [18]:
## set your network proxy
os.environ["http_proxy"] = "http://127.0.0.1:7993"
os.environ["https_proxy"] = "http://127.0.0.1:7993"


In [19]:
# 初始化 Earth Engine
MY_PROJECT_ID = 'my-earth-engine-2026'  # 替换为你的 Earth Engine 项目 ID。

try:
    ee.Initialize(project=MY_PROJECT_ID)
except ee.EEException:
    ee.Authenticate()
    ee.Initialize(project=MY_PROJECT_ID)
print("GEE Initialized successfully with project ID:", MY_PROJECT_ID) 


GEE Initialized successfully with project ID: my-earth-engine-2026


In [20]:
# 参数设置
BOUNDARY_LOCAL_PATH = Path('data/boundary/ygp_region.gpkg')
TILES_LOCAL_PATH = Path('data/boundary/ygp_tiles_100km_buf10km.gpkg')

TARGET_YEAR = 2020
START_ID = 1    # tile id, !! start_id should smaller than end_id
END_ID = 3
EXPORT_FOLDER = 'gee-down'  

BANDS_S2 = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12'] 
MAX_CLOUD_PERCENT = 80  # maximum cloud percentage for image filtering.
CS_THRESHOLD = 0.65     # threshold of Cloud Score+  


#### 1. 读取本地矢量并筛选瓦片


In [21]:
study_area = geemap.vector_to_ee(str(BOUNDARY_LOCAL_PATH))
tiles = geemap.vector_to_ee(str(TILES_LOCAL_PATH))

selected_tiles = (
    tiles.filter(ee.Filter.gte('tile_id', START_ID))
    .filter(ee.Filter.lte('tile_id', END_ID))
    .sort('tile_id_num')
) 
selected_tiles


#### 2. 筛选年度影像并根据Cloud Score+信息去云

In [22]:
s2_annual = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filter(ee.Filter.calendarRange(TARGET_YEAR, TARGET_YEAR, 'year'))
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_CLOUD_PERCENT))
    .filterBounds(study_area)
)

cs_plus_annual = (
    ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
    .filter(ee.Filter.calendarRange(TARGET_YEAR, TARGET_YEAR, 'year'))
    .filterBounds(study_area)
)

# Link scores by system:index and mask unclear pixels.
def mask_cs_plus(image):
    clear_mask = image.select('cs').gte(CS_THRESHOLD)
    return image.updateMask(clear_mask)

s2_with_cs = s2_annual.linkCollection(cs_plus_annual, ['cs'])
s2_masked_annual = s2_with_cs.map(mask_cs_plus).select(BANDS_S2)
s2_masked_annual 


#### 3. 年度数据合成


In [23]:
#  Define seasonal filters.
summer_filter = ee.Filter.calendarRange(9, 11, 'month')
other_months_filter = ee.Filter.Or(
    ee.Filter.calendarRange(1, 8, 'month'),
    ee.Filter.calendarRange( 12,12, 'month'),
)

#  Create median composites.
s2_summer = s2_masked_annual.filter(summer_filter).median()
s2_other_months = s2_masked_annual.filter(other_months_filter).median()

#  Fill summer gaps with observations from other months.
combined_img = s2_summer.unmask(s2_other_months)

#  Convert to uint16 and fill remaining gaps with zero.
final_img = combined_img.toUint16().unmask(0) 


In [24]:
source_summer = ee.Image.constant(1).updateMask(s2_summer.select('B2').mask())
source_other = ee.Image.constant(2).updateMask(s2_other_months.select('B2').mask())

# 0：无数据；1：夏季；2：其他月份。与影像采用相同的填补顺序。
source_map = source_summer.unmask(source_other).unmask(0).toByte()  


#### 4. 数据可视化

In [25]:
map = geemap.Map()
map.centerObject(study_area, 5)

vis_true = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}
vis_false = {'bands': ['B8', 'B4', 'B3'], 'min': 0, 'max': 3000}
source_vis = {
    'min': 0,
    'max': 2,
    'palette': ['000000', '00FF00', 'FFFF00'],
}
# map.addLayer(s2_summer.clipToCollection(study_area), vis_false, '夏季假彩色', False)
# map.addLayer(final_img.clipToCollection(study_area), vis_true, '合成真彩色', False)
map.addLayer(final_img.clipToCollection(study_area), vis_false, '合成假彩色', True)
map.addLayer(source_map.clipToCollection(study_area), source_vis, '像元来源', True)
map.addLayer(selected_tiles, {'color': 'FF0000'}, '待导出瓦片', True)
display(map) 


Map(center=[25.5176044764526, 102.98597828581168], controls=(WidgetControl(options=['position', 'transparent_b…

#### 5. 数据导出   

In [ ]:
# tile_features = selected_tiles.getInfo()['features']
# print(f"待导出瓦片数：{len(tile_features)}")

# for tile_feature in tile_features:
#     properties = tile_feature['properties']
#     tile_id = str(properties['tile_id']).zfill(3)
#     tile_crs = str(properties['proj']).replace(' ', '')
#     tile_geometry = ee.Geometry(tile_feature['geometry'])
#     export_name = f'tibet_s2_{TARGET_YEAR}_tile_{tile_id}'

#     task = ee.batch.Export.image.toDrive(
#         image=final_img,
#         description=export_name,
#         folder=EXPORT_FOLDER,
#         fileNamePrefix=export_name,
#         region=tile_geometry,
#         crs=tile_crs,
#         scale=10,
#         maxPixels=1e13,
#         fileFormat='GeoTIFF',
#     )
#     task.start()
#     print(f"已提交瓦片 {tile_id} 的导出任务, Task ID: {task.id}")
# print("导出任务提交结束，请到 Earth Engine Tasks 查看进度。")

待导出瓦片数：3
已提交瓦片 001 的导出任务, Task ID: B2OCOLV2IMTLHK5EZF2VVYH6
导出任务提交结束，请到 Earth Engine Tasks 查看进度。
